# Simultaneous interventions: public example
This notebook shows the components directly. The shell script uses the same estimators,
tuning, split, evaluation and persistence code. It is not necessary to use the runner.

Small demonstration settings, not publication evidence.
Patient splits are fixed across evaluation initialization seeds. Those seeds measure
optimization variability, not independent population uncertainty. Counterfactual metrics
here require simulation truth; observational outcomes alone do not supply true CATE.


In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import torch
from sklearn.base import clone

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ANALYSIS = ROOT / 'analyses' / 'simultaneous_interventions'
config = json.loads((ANALYSIS / 'config.json').read_text())
spaces = json.loads((ANALYSIS / 'tuning.json').read_text())
OUTPUT = (ANALYSIS / config['defaults']['output']).resolve()
torch.set_num_threads(1)

RUN_EXPERIMENTS = False  # False: load and plot saved results only.
SCENARIOS = list(config.get('scenarios', {'default': {}}))
# To run a subset, edit SCENARIOS, e.g. SCENARIOS = [SCENARIOS[0]].
print('Scenarios:', len(SCENARIOS), 'Results:', OUTPUT)


## Estimators and components
Constructors are visible here. Configuration supplies the displayed architecture and training parameters.

In [ ]:
from catelib.methods.s_learner import SLearner
from catelib.methods.compositional_regression import CompositionalRegression
from catelib.datasets import composite, diabetes
from catelib.data import split_data
from catelib.tuning import tune
from catelib.metrics.effects import evaluate, evaluate_interactions, evaluate_policy
from catelib.results import experiment_folder, completed_model, save_evaluation
from catelib.plotting import load_results, plot_comparison

# Actual estimator objects; architectures remain method-specific and editable.
models = {'s': SLearner(), 'ridge': CompositionalRegression()}


In [ ]:
print(json.dumps({'parameters': config['parameters'], 'training': config['training']}, indent=2))

## Data, tuning, training and evaluation
The following ordinary loops run the selected scenarios, methods and initialization seeds.
Every fit is explicit. Tuning sees training and validation patients only. The best parameters
are frozen across evaluation seeds. Saved, checksum-verified evaluations are skipped.
No hidden `run_analysis()` or second training implementation is used.


In [ ]:
if RUN_EXPERIMENTS:
    for scenario in SCENARIOS:
        settings = dict(config['defaults'], **config.get('scenarios', {}).get(scenario, {}))
        dataset_name = settings.get('dataset', 'composite')
        provider = {'composite': composite, 'diabetes': diabetes}[dataset_name]
        data_seed = settings.get('data_seed', 0)
        data = provider(data_seed=data_seed, **settings.get('dataset_params', {}))
        splits = split_data(data, **dict({'seed': data_seed+2}, **settings.get('split', {})))
        train, validation, test = [data.subset(splits[k]) for k in ('train', 'validation', 'test')]
        evaluation = settings.get('evaluation', {})
        missing = evaluation.get('missing_outcome_rate', 0)
        if missing:
            X, Y, T = train
            mask = np.random.default_rng(data_seed+4).random(Y.shape) < missing
            train = (X, np.where(mask, np.nan, Y), T)
        print(scenario, {k: len(splits[k]) for k in ('train','validation','test')})
        reference = np.asarray(evaluation.get('reference', [0 if t=='binary' else .5 for t in data.treatment_types]))
        interventions = data.truth.intervention_grid(evaluation.get('doses', [.25,.75]))

        for name in config['methods'][scenario]:
            model = clone(models[name])
            available = model.get_params(deep=True)
            parameters = dict(config['parameters'][name], **config.get('scenario_parameters', {}).get(scenario, {}).get(name, {}))
            for key in available:
                if key.split('__')[-1] == 'outcome_types': parameters[key] = data.outcome_types
                if key.split('__')[-1] == 'random_state': parameters[key] = 0
                for setting, value in config['training'][name].items():
                    if key.split('__')[-1] == setting: parameters[key] = value
            model.set_params(**parameters)
            search_options = config.get('scenario_tuning', {}).get(scenario, {}).get(name, spaces[name])
            folder = experiment_folder(OUTPUT, method=name, dataset=dataset_name, data_seed=data_seed,
                train=train, validation=validation, test=test, estimator=model, tuning=search_options,
                evaluation=evaluation, metrics=settings['metrics'], scenario=scenario)
            np.savez_compressed(folder/'data.npz', X=data.X, Y=data.Y, treatment=data.treatment, **splits)

            search = tune(model, train=train, validation=validation, save_dir=folder/'tuning', **search_options)
            print(name, 'Selected parameters:', search.best_params, 'Validation loss:', search.best_score)
            for step, seed in enumerate(settings['seeds'], 1):
                destination = folder / f'seed-{seed}'
                if completed_model(destination, 'evaluation') is not None:
                    print(name, seed, 'cached')
                    continue
                fitted = clone(model).set_params(**search.best_params)
                fitted.set_params(**{k: seed for k in fitted.get_params(deep=True) if k.split('__')[-1]=='random_state'})
                started = time.perf_counter()
                fitted.fit(train[0], train[1], treatment=train[2], validation_data=validation)
                fit_seconds = time.perf_counter() - started
                measured = {}
                for metric in settings['metrics']:
                    try:
                        if metric == 'effects':
                            measured[metric] = evaluate(fitted, data.truth, test[0], interventions, reference,
                                holdout_pattern=evaluation.get('holdout_pattern'), allow_unsupported=name in ('t','dr','tarnet','cfr'))
                        elif metric == 'interactions':
                            measured[metric] = evaluate_interactions(fitted, data.truth, test[0], reference, np.max(interventions,axis=0))
                        elif metric == 'policy':
                            measured[metric] = evaluate_policy(fitted, data.truth, test[0], interventions,
                                evaluation.get('utility_weights', np.ones(data.Y.shape[1])), evaluation.get('treatment_costs'))
                    except ValueError as error:
                        if name not in ('t','dr','tarnet','cfr'): raise
                        measured[metric] = {'status':'unsupported', 'reason':str(error)}
                save_evaluation(destination, model=fitted, data=data, splits=splits,
                    interventions=interventions, reference=reference, metrics=measured,
                    method=name, scenario=scenario, seed=seed, fit_seconds=fit_seconds, best_params=search.best_params)
                bar = '#' * (20 * step // len(settings['seeds']))
                print(f'{name} [{bar:<20}] {step}/{len(settings["seeds"])} evaluations')
else:
    print('Training disabled. The next cells load existing results only.')


## Final tables and plots from saved results
The shell script and this notebook use the same reporting function. It reads saved
results only, exports CSV tables and PNG figures, and displays them here. Training
above remains explicit and can be disabled independently. Reports are separated by
scenario and outcome; missing coverage and incomplete runs are shown before scores.
CATE/mean comparisons use common supported contrasts across all expected methods.
Policy scores are finite-grid oracle simulation values, not observational estimates.
Initialization-seed variation is not a population confidence interval.


In [ ]:
from catelib.plotting import report_results

reports = {}
for scenario in SCENARIOS:
    settings = dict(config['defaults'], **config.get('scenarios', {}).get(scenario, {}))
    expected_methods = config['methods'][scenario]
    reports[scenario] = report_results(
        OUTPUT, scenario=scenario, expected_methods=expected_methods,
        expected_seeds=settings['seeds'], show=True,
    )
    print('Saved report:', reports[scenario]['output'])
